# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sehreen-Atta/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [10]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


This is a clustering task. My lane groups content items into 
archetypes based on shared characteristics (content type, intent, 
structure, and SEO signals), rather than predicting a single known 
label. There's no ground-truth "correct" archetype for each item — 
clustering discovers natural groupings from the data itself.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [12]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

Clustering doesn't use a predefined target label — instead, I'm 
defining archetypes based on a set of features that describe each 
piece of content's structure and characteristics. The "proxy" for 
an archetype is the combination of these feature values, not a 
single column.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

A defensible metric here is silhouette score — it measures how 
similar each content item is to others in its own cluster versus 
items in other clusters, with values closer to 1 meaning tighter, 
more distinct clusters. Running KMeans (k=4) on this lane's features 
(word_count, char_count, impressions_90d, clicks_90d, pageviews_90d, 
sessions_90d, engaged_sessions_90d) produced a silhouette score of 
0.537, indicating the clusters are reasonably well-separated rather 
than arbitrary. Practically, "good" also means each cluster maps to 
a distinguishable content strategy — for example, one archetype 
might show declining trend_pct and low ai_traffic_pct (a refresh 
candidate), while another shows stable trend_direction and high 
engaged_sessions_90d (a healthy performer that needs no action).

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = ['word_count', 'char_count', 'impressions_90d', 'clicks_90d', 
            'pageviews_90d', 'sessions_90d', 'engaged_sessions_90d']

X = df[features].dropna()
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

score = silhouette_score(X_scaled, labels)
print(f"Silhouette score: {score:.3f}")

Silhouette score: 0.537


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [15]:
df[['content_id', 'client_id', 'content_type', 'main_intent', 
    'trend_direction', 'trend_pct']].head()

,content_id,client_id,content_type,main_intent,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,down,-34.7


One row = one content item, uniquely identified by content_id, 
tied to a specific client (client_id). The dataset has 30,000 rows 
and 44 columns, spanning SEO signals (search_volume, competition, 
cpc), content attributes (content_type, main_intent, word_count), 
performance metrics (clicks_90d, sessions_90d, engaged_sessions_90d), 
and trend indicators (trend_direction, trend_pct, position_tier).

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [17]:
df[['trend_pct', 'search_volume', 'position_tier', 'engaged_sessions_90d']].corr(numeric_only=True)

,trend_pct,search_volume,engaged_sessions_90d
trend_pct,1.000000,0.002515,0.001598
search_volume,0.002515,1.000000,-0.009712
engaged_sessions_90d,0.001598,-0.009712,1.000000


A fixed rule (e.g., "if trend_pct < -10%, flag for refresh") only 
looks at one signal at a time and misses how features interact. 
Two content items could have the same trend_pct but very different 
situations — one with high search_volume and declining position_tier 
(worth refreshing), another with low search_volume and stable 
engaged_sessions_90d (not worth the effort). Clustering considers 
all these signals together, surfacing archetypes a single-column 
rule would conflate or miss entirely.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.